# 日本語BM25（Qdrant + FastEmbed）：分かち書きなし vs SudachiPy

記事: 「QdrantのBM25を日本語で使う：分かち書きなしと比べて、SudachiPyで何が変わるか」の検証ノートブックです。

比較する方式

| 記号 | 前処理 | avg_len |
| --- | --- | --- |
| A | なし（テキストをそのまま`Qdrant/bm25`へ） | FastEmbed既定値(256) |
| A2 | なし | 実データの平均（B/Cと同じ値） |
| B | Sudachi名詞の`normalized_form()`（lowerなし）をhex化 | 実データの平均 |
| C | Bに`.lower()`を追加 | 実データの平均 |
| C256 | Cと同じ | 既定値(256) |

- A と A2 の差 → avg_len の影響
- A2 と C の差 → 分かち書きの影響（avg_lenをそろえた比較）
- B と C の差 → 大文字小文字のぶれの影響
- C と C256 の差 → avg_len の影響（Sudachi側）

注意: 文書6件は記事の表（ID・内容）から**再作成**したテキストです。元のノートブックの文面とは異なるため、スコアは記事の値と一致しません。

## 0. セットアップ

In [1]:
# バージョンは記事の検証環境に合わせています（SudachiDict-coreは日付版のため固定していません）
!pip -q install "langchain-qdrant==1.1.0" "qdrant-client==1.19.0" "fastembed==0.8.0" "SudachiPy==0.6.11" sudachidict_core

In [2]:
from importlib.metadata import version
for p in ["langchain-qdrant", "qdrant-client", "fastembed", "SudachiPy", "SudachiDict-core"]:
    try:
        print(p, version(p))
    except Exception as e:
        print(p, "?", e)

langchain-qdrant 1.1.0
qdrant-client 1.19.0
fastembed 0.8.0
SudachiPy 0.6.11
SudachiDict-core 20260723


## 1. 評価データ（6件・3クエリ）

In [3]:
from langchain_core.documents import Document

DOCS = [
    ("engineer-001", "Pythonを3年間使用し、AWS Lambdaを利用したサーバーレスAPIの開発経験があります。"),
    ("engineer-002", "JavaとSpring Bootを使用し、MySQLとDockerを利用した業務システムの開発経験があります。"),
    ("engineer-003", "C++とCUDAを使用し、TensorRTとJetson Orinを利用したエッジAIの画像認識の開発経験があります。"),
    ("engineer-004", "TypeScriptとReactを使用し、Next.jsとAWSを利用したWebアプリの開発経験があります。"),
    ("engineer-005", "Pythonを使用し、LangChainとQdrantを利用したRAGとローカルLLMの開発経験があります。"),
    ("engineer-006", "COBOLを使用し、銀行の勘定系システムの保守と開発経験があります。"),
]
documents = [Document(page_content=t, metadata={"id": i}) for i, t in DOCS]

# (クエリ, 正解ID)
QUERIES = [
    ("Python AWS Lambdaを利用した開発経験", "engineer-001"),
    ("C++ TensorRT Jetson 画像AI開発", "engineer-003"),
    ("Python Qdrant LangChain RAGシステム", "engineer-005"),
]

## 2. Sudachi tokenizer

In [4]:
from sudachipy import Dictionary, SplitMode


class SudachiBM25Tokenizer:
    """SplitMode.Cで分割し、名詞のnormalized_form()を返す。lower=Trueで小文字化"""

    def __init__(self, split_mode=SplitMode.C, target_pos=None, lower=True):
        self.tokenizer = Dictionary().create()
        self.split_mode = split_mode
        self.target_pos = target_pos or {"名詞"}
        self.lower = lower

    def __call__(self, text: str) -> list[str]:
        tokens = []
        for m in self.tokenizer.tokenize(text, self.split_mode):
            if m.part_of_speech()[0] not in self.target_pos:
                continue
            token = m.normalized_form().strip()
            if self.lower:
                token = token.lower()
            if token:
                tokens.append(token)
        return tokens


tok_nolower = SudachiBM25Tokenizer(lower=False)   # B
tok_lower = SudachiBM25Tokenizer(lower=True)      # C

for doc_id, text in DOCS:
    print(doc_id, tok_nolower(text))

engineer-001 ['python', '3', '年間', '使用', 'AWS', 'lambda', '利用', 'サーバーレス', 'API', '開発', '経験']
engineer-002 ['Java', 'スプリング', 'ブート', '使用', 'mysql', 'Docker', '利用', '業務', 'システム', '開発', '経験']
engineer-003 ['c', 'cuda', '使用', 'tensorrt', 'jetson', 'orin', '利用', 'エッジAI', '画像', '認識', '開発', '経験']
engineer-004 ['typescript', 'react', '使用', 'ネクスト', 'js', 'AWS', '利用', 'ウェブ', 'アプリ', '開発', '経験']
engineer-005 ['python', '使用', 'langchain', 'qdrant', '利用', 'rag', 'ローカル', 'LLM', '開発', '経験']
engineer-006 ['COBOL', '使用', '銀行', '勘定', 'システム', '保守', '開発', '経験']


In [5]:
# 「Python」のぶれの確認: lowerなしだと文脈で 'Python' / 'python' が混在するか
for text in [d.page_content for d in documents] + [q for q, _ in QUERIES]:
    toks = [t for t in tok_nolower(text) if t.lower() == "python"]
    if toks:
        print(toks, "<-", text)

['python'] <- Pythonを3年間使用し、AWS Lambdaを利用したサーバーレスAPIの開発経験があります。
['python'] <- Pythonを使用し、LangChainとQdrantを利用したRAGとローカルLLMの開発経験があります。
['Python'] <- Python AWS Lambdaを利用した開発経験
['Python'] <- Python Qdrant LangChain RAGシステム


In [6]:
# avg_len（実データの平均トークン数）。BとCは同じ値になるはず
lens_b = [len(tok_nolower(d.page_content)) for d in documents]
lens_c = [len(tok_lower(d.page_content)) for d in documents]
AVG_LEN = sum(lens_c) / len(lens_c)
print("lens:", lens_c, "avg_len:", AVG_LEN, "(B側と一致:", lens_b == lens_c, ")")

lens: [11, 11, 12, 11, 10, 8] avg_len: 10.5 (B側と一致: True )


## 3. SparseEmbeddingsの定義

In [8]:
from langchain_qdrant import FastEmbedSparse, SparseEmbeddings, SparseVector


class JapaneseBM25SparseEmbeddings(SparseEmbeddings):
    """SudachiPy + FastEmbedSparse(Qdrant/bm25)。トークンはUTF-8のhex文字列にして渡す"""

    def __init__(self, tokenize, avg_len, *, k=1.5, b=0.75):
        self.tokenize = tokenize
        self.bm25 = FastEmbedSparse(
            model_name="Qdrant/bm25",
            disable_stemmer=True,
            k=k, b=b,
            avg_len=avg_len,
            token_max_length=512,   # hex化で文字数が増える
        )

    @staticmethod
    def _encode_token(token: str) -> str:
        return "t" + token.encode("utf-8").hex()

    def preprocess(self, text: str) -> str:
        return " ".join(self._encode_token(t) for t in self.tokenize(text))

    def embed_documents(self, texts):
        return self.bm25.embed_documents([self.preprocess(t) for t in texts])

    def embed_query(self, text):
        return self.bm25.embed_query(self.preprocess(text))


def make_raw(avg_len=None):
    """分かち書きなし。avg_len=Noneならfastembedの既定値(256)"""
    kwargs = dict(model_name="Qdrant/bm25", disable_stemmer=True, k=1.5, b=0.75)
    if avg_len is not None:
        kwargs["avg_len"] = avg_len
    return FastEmbedSparse(**kwargs)


EMBEDDINGS = {
    "A":    make_raw(),                                                      # 分かち書きなし / avg_len既定
    "A2":   make_raw(AVG_LEN),                                               # 分かち書きなし / avg_len実データ
    "B":    JapaneseBM25SparseEmbeddings(tok_nolower, AVG_LEN),              # Sudachi / lowerなし
    "C":    JapaneseBM25SparseEmbeddings(tok_lower, AVG_LEN),                # Sudachi / lowerあり
    "C256": JapaneseBM25SparseEmbeddings(tok_lower, 256.0),                  # Sudachi / lowerあり / avg_len既定
}
print(list(EMBEDDINGS))

['A', 'A2', 'B', 'C', 'C256']


## 4. Qdrantに登録

Sparse Vectorのコレクションには`Modifier.IDF`を指定します（IDFはQdrant側で計算されるため）。

In [9]:
from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore, RetrievalMode

client = QdrantClient(":memory:")
SPARSE_NAME = "bm25"
STORES = {}

for name, emb in EMBEDDINGS.items():
    col = f"bm25_{name}"
    client.create_collection(
        collection_name=col,
        sparse_vectors_config={SPARSE_NAME: models.SparseVectorParams(modifier=models.Modifier.IDF)},
    )
    store = QdrantVectorStore(
        client=client,
        collection_name=col,
        sparse_embedding=emb,
        sparse_vector_name=SPARSE_NAME,
        retrieval_mode=RetrievalMode.SPARSE,
    )
    store.add_documents(documents)
    STORES[name] = store

print("registered:", list(STORES))

registered: ['A', 'A2', 'B', 'C', 'C256']


## 5. 検索と評価

指標: 正解文書の順位 / 1位のスコア / クエリと正解文書の**一致トークン数**（Sparse Vectorの添字の積集合）。
スパース検索では、クエリと1語も一致しない文書は返らないため、その場合は「圏外」と表示します。

In [10]:
import pandas as pd

DOC_TEXT = {d.metadata["id"]: d.page_content for d in documents}


def matched_count(emb, query, doc_text):
    q = emb.embed_query(query)
    d = emb.embed_documents([doc_text])[0]
    return len(set(q.indices) & set(d.indices))


rows = []
rankings = {}
for name, store in STORES.items():
    for query, gold in QUERIES:
        res = store.similarity_search_with_score(query, k=len(documents))
        ids = [doc.metadata["id"] for doc, _ in res]
        scores = [s for _, s in res]
        rankings[(name, query)] = list(zip(ids, scores))
        rank = ids.index(gold) + 1 if gold in ids else None
        rows.append({
            "方式": name,
            "クエリ": query,
            "正解": gold,
            "正解の順位": rank if rank else "圏外",
            "1位": ids[0] if ids else "-",
            "1位スコア": round(scores[0], 3) if scores else None,
            "正解のスコア": round(scores[rank - 1], 3) if rank else None,
            "一致トークン数(正解)": matched_count(EMBEDDINGS[name], query, DOC_TEXT[gold]),
        })

df = pd.DataFrame(rows)
pd.set_option("display.width", 200, "display.max_colwidth", 60)
df

,方式,クエリ,正解,正解の順位,1位,1位スコア,正解のスコア,一致トークン数(正解)
0,A,Python AWS Lambdaを利用した開発経験,engineer-001,1,engineer-001,2.774,2.774,1
1,A,C++ TensorRT Jetson 画像AI開発,engineer-003,1,engineer-003,2.765,2.765,1
2,A,Python Qdrant LangChain RAGシステム,engineer-005,圏外,-,NaN,NaN,0
3,A2,Python AWS Lambdaを利用した開発経験,engineer-001,1,engineer-001,2.270,2.270,1
4,A2,C++ TensorRT Jetson 画像AI開発,engineer-003,1,engineer-003,2.135,2.135,1
5,A2,Python Qdrant LangChain RAGシステム,engineer-005,圏外,-,NaN,NaN,0
6,B,Python AWS Lambdaを利用した開発経験,engineer-001,1,engineer-001,2.897,2.897,5
7,B,C++ TensorRT Jetson 画像AI開発,engineer-003,1,engineer-003,5.859,5.859,5
8,B,Python Qdrant LangChain RAGシステム,engineer-005,1,engineer-005,4.723,4.723,3
9,C,Python AWS Lambdaを利用した開発経験,engineer-001,1,engineer-001,3.905,3.905,6


In [11]:
# 方式 × クエリ の順位表（記事の表1用）
df.pivot(index="クエリ", columns="方式", values="正解の順位")[list(EMBEDDINGS)]

方式,A,A2,B,C,C256
クエリ,,,,,
C++ TensorRT Jetson 画像AI開発,1,1,1,1,1
Python AWS Lambdaを利用した開発経験,1,1,1,1,1
Python Qdrant LangChain RAGシステム,圏外,圏外,1,1,1


In [12]:
# 方式 × クエリ の一致トークン数（記事の表2用）
df.pivot(index="クエリ", columns="方式", values="一致トークン数(正解)")[list(EMBEDDINGS)]

方式,A,A2,B,C,C256
クエリ,,,,,
C++ TensorRT Jetson 画像AI開発,1,1,5,5,5
Python AWS Lambdaを利用した開発経験,1,1,5,6,6
Python Qdrant LangChain RAGシステム,0,0,3,4,4


In [13]:
# 全順位の詳細（2位以降のスコアも見る）
for (name, query), ranking in rankings.items():
    print(f"[{name}] {query}")
    for i, (doc_id, score) in enumerate(ranking, 1):
        print(f"   {i}. {doc_id}  {score:.3f}")

[A] Python AWS Lambdaを利用した開発経験
   1. engineer-001  2.774
[A] C++ TensorRT Jetson 画像AI開発
   1. engineer-003  2.765
[A] Python Qdrant LangChain RAGシステム
[A2] Python AWS Lambdaを利用した開発経験
   1. engineer-001  2.270
[A2] C++ TensorRT Jetson 画像AI開発
   1. engineer-003  2.135
[A2] Python Qdrant LangChain RAGシステム
[B] Python AWS Lambdaを利用した開発経験
   1. engineer-001  2.897
   2. engineer-004  1.389
   3. engineer-005  0.398
   4. engineer-002  0.381
   5. engineer-003  0.366
   6. engineer-006  0.166
[B] C++ TensorRT Jetson 画像AI開発
   1. engineer-003  5.859
   2. engineer-006  0.083
   3. engineer-005  0.076
   4. engineer-004  0.073
   5. engineer-002  0.073
   6. engineer-001  0.073
[B] Python Qdrant LangChain RAGシステム
   1. engineer-005  4.723
   2. engineer-006  1.153
   3. engineer-002  1.008
[C] Python AWS Lambdaを利用した開発経験
   1. engineer-001  3.905
   2. engineer-005  1.450
   3. engineer-004  1.389
   4. engineer-002  0.381
   5. engineer-003  0.366
   6. engineer-006  0.166
[C] C++ TensorRT Jetso

## 6. 補助確認

### 6-1. Sparse Vectorの値がBM25のTF式と一致するか（方式C）

In [14]:
emb = EMBEDDINGS["C"]
doc = DOC_TEXT["engineer-001"]
vec = emb.embed_documents([doc])[0]
n = len(tok_lower(doc))
k, b = 1.5, 0.75
expected = 1 * (k + 1) / (1 + k * (1 - b + b * n / AVG_LEN))
print("doc_len:", n, "avg_len:", AVG_LEN)
print("vector values:", sorted(set(round(v, 6) for v in vec.values)))
print("expected(count=1):", round(expected, 6))

doc_len: 11 avg_len: 10.5
vector values: [0.979021]
expected(count=1): 0.979021


### 6-2. クエリ側のSparse Vectorの値（出現回数が使われないことの確認）

In [15]:
q = EMBEDDINGS["C"].embed_query("Python Python AWS")
print(q.indices, q.values)

[734446929, 566669935] [1.0, 1.0]


### 6-3. 方式Bで、クエリの`Python`が文書001に一致しているか

In [16]:
for name in ["B", "C"]:
    emb = EMBEDDINGS[name]
    q_idx = set(emb.embed_query("Python AWS Lambda 開発経験").indices)
    d_idx = set(emb.embed_documents([DOC_TEXT["engineer-001"]])[0].indices)
    py_idx = emb.embed_query("Python").indices
    print(name, "Pythonが一致:", bool(set(py_idx) & d_idx), "/ 共通添字数:", len(q_idx & d_idx))

B Pythonが一致: False / 共通添字数: 4
C Pythonが一致: True / 共通添字数: 5


### 6-4. 方式Aは日本語をどう扱っているか（FastEmbedに渡した結果の添字数）

In [17]:
a = EMBEDDINGS["A"]
for doc_id, text in DOCS[:3]:
    v = a.embed_documents([text])[0]
    print(doc_id, "非ゼロ要素:", len(v.indices), "| Sudachi名詞数:", len(tok_lower(text)))

engineer-001 非ゼロ要素: 3 | Sudachi名詞数: 11
engineer-002 非ゼロ要素: 3 | Sudachi名詞数: 11
engineer-003 非ゼロ要素: 4 | Sudachi名詞数: 12


## 7. 結果の読み方

- A と A2 で順位が変わる → avg_len の影響が出ている
- A2 と C で順位・一致トークン数が違う → 分かち書きの影響（avg_lenをそろえた比較）
- B と C で違う → 大文字小文字のぶれの影響
- C と C256 で違う → Sudachi側でも avg_len の影響が出る
- **どの方式でも順位が同じ** → 「この6件・3クエリでは差が出なかった」。その場合、記事の主張は
  「固有名詞は分かち書きなしでも拾える。差が出るのは日本語の語（開発、経験など）」のように結果に合わせて修正する
- 文書6件・正解1件のため、順位の一致は精度の証拠にならない。再現率の評価は数十件以上に拡充してから

## 8. 追加検証：日本語の語だけのクエリ

ここまでのクエリは、`Python`や`TensorRT`のような英字の固有名詞を含んでいました。方式Aが検索できたのは、これが理由だった可能性があります。
ここでは、**日本語の語だけ**のクエリで、5方式を比べます。

| クエリ | 正解 |
| --- | --- |
| サーバーレスなAPIの開発経験 | engineer-001 |
| 銀行の勘定系システムの保守 | engineer-006 |
| 画像認識の開発経験 | engineer-003 |
| ローカルLLMを利用した開発 | engineer-005 |

注意: 「開発」「経験」「使用」「利用」は全6文書に出るため、IDFが低く、スコアにはほとんど効きません。

In [18]:
EXTRA_QUERIES = [
    ("サーバーレスなAPIの開発経験", "engineer-001"),
    ("銀行の勘定系システムの保守", "engineer-006"),
    ("画像認識の開発経験", "engineer-003"),
    ("ローカルLLMを利用した開発", "engineer-005"),
]


def evaluate(queries):
    rows, rankings = [], {}
    for name, store in STORES.items():
        for query, gold in queries:
            res = store.similarity_search_with_score(query, k=len(documents))
            ids = [doc.metadata["id"] for doc, _ in res]
            scores = [s for _, s in res]
            rankings[(name, query)] = list(zip(ids, scores))
            rank = ids.index(gold) + 1 if gold in ids else None
            rows.append({
                "方式": name,
                "クエリ": query,
                "正解": gold,
                "正解の順位": rank if rank else "圏外",
                "返った件数": len(ids),
                "1位": ids[0] if ids else "-",
                "1位スコア": round(scores[0], 3) if scores else None,
                "一致トークン数(正解)": matched_count(EMBEDDINGS[name], query, DOC_TEXT[gold]),
            })
    return pd.DataFrame(rows), rankings


df_extra, rankings_extra = evaluate(EXTRA_QUERIES)
df_extra

,方式,クエリ,正解,正解の順位,返った件数,1位,1位スコア,一致トークン数(正解)
0,A,サーバーレスなAPIの開発経験,engineer-001,圏外,0,-,NaN,0
1,A,銀行の勘定系システムの保守,engineer-006,圏外,0,-,NaN,0
2,A,画像認識の開発経験,engineer-003,圏外,0,-,NaN,0
3,A,ローカルLLMを利用した開発,engineer-005,圏外,0,-,NaN,0
4,A2,サーバーレスなAPIの開発経験,engineer-001,圏外,0,-,NaN,0
5,A2,銀行の勘定系システムの保守,engineer-006,圏外,0,-,NaN,0
6,A2,画像認識の開発経験,engineer-003,圏外,0,-,NaN,0
7,A2,ローカルLLMを利用した開発,engineer-005,圏外,0,-,NaN,0
8,B,サーバーレスなAPIの開発経験,engineer-001,1,6,engineer-001,3.161,4
9,B,銀行の勘定系システムの保守,engineer-006,1,2,engineer-006,6.329,4


In [20]:
# 正解の順位（方式 × クエリ）
df_extra.pivot(index="クエリ", columns="方式", values="正解の順位")[list(EMBEDDINGS)]

方式,A,A2,B,C,C256
クエリ,,,,,
サーバーレスなAPIの開発経験,圏外,圏外,1,1,1
ローカルLLMを利用した開発,圏外,圏外,1,1,1
画像認識の開発経験,圏外,圏外,1,1,1
銀行の勘定系システムの保守,圏外,圏外,1,1,1


In [19]:
# 一致トークン数（方式 × クエリ）
df_extra.pivot(index="クエリ", columns="方式", values="一致トークン数(正解)")[list(EMBEDDINGS)]

方式,A,A2,B,C,C256
クエリ,,,,,
サーバーレスなAPIの開発経験,0,0,4,4,4
ローカルLLMを利用した開発,0,0,4,4,4
画像認識の開発経験,0,0,4,4,4
銀行の勘定系システムの保守,0,0,4,4,4


In [21]:
# 返った件数（方式 × クエリ）。Sparse検索は、1語も一致しない文書は返さない
df_extra.pivot(index="クエリ", columns="方式", values="返った件数")[list(EMBEDDINGS)]

方式,A,A2,B,C,C256
クエリ,,,,,
サーバーレスなAPIの開発経験,0,0,6,6,6
ローカルLLMを利用した開発,0,0,6,6,6
画像認識の開発経験,0,0,6,6,6
銀行の勘定系システムの保守,0,0,2,2,2


In [22]:
# 方式CとBの全順位（2位以下のスコアも見る）
for (name, query), ranking in rankings_extra.items():
    if name not in ("B", "C"):
        continue
    print(f"[{name}] {query}")
    for i, (doc_id, score) in enumerate(ranking, 1):
        print(f"   {i}. {doc_id}  {score:.3f}")

[B] サーバーレスなAPIの開発経験
   1. engineer-001  3.161
   2. engineer-006  0.166
   3. engineer-005  0.151
   4. engineer-004  0.145
   5. engineer-002  0.145
   6. engineer-003  0.139
[B] 銀行の勘定系システムの保守
   1. engineer-006  6.329
   2. engineer-002  1.008
[B] 画像認識の開発経験
   1. engineer-003  3.034
   2. engineer-006  0.166
   3. engineer-005  0.151
   4. engineer-004  0.145
   5. engineer-002  0.145
   6. engineer-001  0.145
[B] ローカルLLMを利用した開発
   1. engineer-005  3.471
   2. engineer-004  0.309
   3. engineer-001  0.309
   4. engineer-002  0.309
   5. engineer-003  0.296
   6. engineer-006  0.083
[C] サーバーレスなAPIの開発経験
   1. engineer-001  3.161
   2. engineer-006  0.166
   3. engineer-005  0.151
   4. engineer-004  0.145
   5. engineer-002  0.145
   6. engineer-003  0.139
[C] 銀行の勘定系システムの保守
   1. engineer-006  6.329
   2. engineer-002  1.008
[C] 画像認識の開発経験
   1. engineer-003  3.034
   2. engineer-006  0.166
   3. engineer-005  0.151
   4. engineer-004  0.145
   5. engineer-002  0.145
   6. engineer-001 

## 9. 方式Aの入力は、FastEmbedの中でどう分けられているか

`fastembed 0.8.0`の`Bm25.raw_embed()`は、次の順で処理します（ソースで確認）。

1. `remove_non_alphanumeric()` … `\w`と空白以外を空白に置き換える
2. `SimpleTokenizer.tokenize()` … 小文字化して、`\w`以外で分割する
3. `_stem()` … `token_max_length`（既定40文字）を超えるトークンを**捨てる**

Pythonの正規表現では、ひらがな・カタカナ・漢字は`\w`に含まれます。そのため、**日本語の文字は区切りにならず**、空白や句読点で挟まれたかたまりが1トークンになります。
下のセルで、これを再現します。

In [23]:
from fastembed.common.utils import remove_non_alphanumeric
from fastembed.sparse.utils.tokenizer import SimpleTokenizer

TOKEN_MAX_LENGTH = 40   # 方式Aは token_max_length を指定していない = FastEmbedの既定値


def fastembed_tokens(text: str, token_max_length: int = TOKEN_MAX_LENGTH):
    """Bm25.raw_embed()（stemmerなし）の前処理を再現する。戻り値: (残るトークン, 捨てられるトークン)"""
    tokens = SimpleTokenizer.tokenize(remove_non_alphanumeric(text))
    kept = [t for t in tokens if len(t) <= token_max_length]
    dropped = [t for t in tokens if len(t) > token_max_length]
    return kept, dropped


for doc_id, text in DOCS:
    kept, dropped = fastembed_tokens(text)
    print(doc_id)
    print("   残る  :", kept)
    if dropped:
        print("   捨てる:", dropped, "(", [len(t) for t in dropped], "文字 )")

engineer-001
   残る  : ['pythonを3年間使用し', 'aws', 'lambdaを利用したサーバーレスapiの開発経験があります']
engineer-002
   残る  : ['javaとspring', 'bootを使用し', 'mysqlとdockerを利用した業務システムの開発経験があります']
engineer-003
   残る  : ['c', 'とcudaを使用し', 'tensorrtとjetson', 'orinを利用したエッジaiの画像認識の開発経験があります']
engineer-004
   残る  : ['typescriptとreactを使用し', 'next', 'jsとawsを利用したwebアプリの開発経験があります']
engineer-005
   残る  : ['pythonを使用し']
   捨てる: ['langchainとqdrantを利用したragとローカルllmの開発経験があります'] ( [42] 文字 )
engineer-006
   残る  : ['cobolを使用し', '銀行の勘定系システムの保守と開発経験があります']


In [24]:
# 再現が正しいかの確認: 再現したトークンのユニーク数 = 方式Aの実際のSparse Vectorの非ゼロ要素数
ok = True
for doc_id, text in DOCS:
    kept, _ = fastembed_tokens(text)
    actual = len(EMBEDDINGS["A"].embed_documents([text])[0].indices)
    flag = "OK" if len(set(kept)) == actual else "NG"
    ok &= (flag == "OK")
    print(doc_id, "再現:", len(set(kept)), "実際:", actual, flag)
print("全文書で一致:", ok)

engineer-001 再現: 3 実際: 3 OK
engineer-002 再現: 3 実際: 3 OK
engineer-003 再現: 4 実際: 4 OK
engineer-004 再現: 3 実際: 3 OK
engineer-005 再現: 1 実際: 1 OK
engineer-006 再現: 2 実際: 2 OK
全文書で一致: True


In [25]:
# 方式Aで、クエリと正解文書の間で一致したトークンの正体
for query, gold in QUERIES + EXTRA_QUERIES:
    q_kept, _ = fastembed_tokens(query)
    d_kept, _ = fastembed_tokens(DOC_TEXT[gold])
    shared = sorted(set(q_kept) & set(d_kept))
    print(f"{query}  -> {gold}")
    print("   クエリのトークン:", q_kept)
    print("   一致:", shared if shared else "なし")

Python AWS Lambdaを利用した開発経験  -> engineer-001
   クエリのトークン: ['python', 'aws', 'lambdaを利用した開発経験']
   一致: ['aws']
C++ TensorRT Jetson 画像AI開発  -> engineer-003
   クエリのトークン: ['c', 'tensorrt', 'jetson', '画像ai開発']
   一致: ['c']
Python Qdrant LangChain RAGシステム  -> engineer-005
   クエリのトークン: ['python', 'qdrant', 'langchain', 'ragシステム']
   一致: なし
サーバーレスなAPIの開発経験  -> engineer-001
   クエリのトークン: ['サーバーレスなapiの開発経験']
   一致: なし
銀行の勘定系システムの保守  -> engineer-006
   クエリのトークン: ['銀行の勘定系システムの保守']
   一致: なし
画像認識の開発経験  -> engineer-003
   クエリのトークン: ['画像認識の開発経験']
   一致: なし
ローカルLLMを利用した開発  -> engineer-005
   クエリのトークン: ['ローカルllmを利用した開発']
   一致: なし


## 10. 追加検証の読み方

- **8の方式Aが「圏外」** → 日本語の語だけのクエリでは、分かち書きなしのBM25は照合できない（空白や句読点で区切られたかたまり全体が1トークンのため）
- **8の方式B・Cで正解が1位** → 分かち書きで、日本語の語が照合される
- **9で「捨てる」に長いトークンが出る** → 文書によっては、方式Aのトークンが`token_max_length`（既定40文字）で捨てられ、Sparse Vectorがほぼ空になる（`engineer-005`など）
- **9の再現が「NG」** → 手元の`fastembed`のバージョンでは処理が違う。バージョンを確認する